In [1]:
import os, pickle, tempfile, requests
import pandas as pd

In [2]:
import sys
sys.path.append("../../training_data")

In [3]:
from utils.utils import Cif
from utils.pocket_utils import Pocket

# Data

In [4]:
ups = (
    ("7gqu", "q14191"), # done
    ("8qni", "Q13191"),
    ("8v81", "P13569"),
    ("8jp0", "P32418"),
    ("7yg5", "Q15878")
    # 8f4s not available
    # 8aq6 dimer not available
    # 8uk6 already included
)

## Process AlphaFold

In [5]:
for holo, up_id in ups:
    name = f"AF-{up_id.upper()}-F1"
    outfname = f"{name.lower()}_out.cif"
    if not os.path.isfile(outfname):
        with tempfile.NamedTemporaryFile(suffix=".cif", mode="w+") as f:
            f.write(requests.get(f"https://alphafold.ebi.ac.uk/files/{name.upper()}-model_v6.cif").content.decode())
            cif = Cif(name.lower(), f.name)
            sel_res = cif.atoms.loc[cif.atoms["B_iso_or_equiv"].astype(float) > 50][["label_asym_id", "label_seq_id"]].drop_duplicates()
            with cif._extended_temp_ciff({
                    "_atom_site": cif.atoms.merge(sel_res).to_dict(orient="list")
                }) as f:
                with open(outfname, "w") as outf:
                    outf.write(f.read())

## Collect results

In [7]:
news = {}

for holo, pdb_id, jobid, jobdir in (
    ("7gqu", "af-q14191-f1", "af-q14191-f1_out_1153095567726851806", "af-q14191-f1_out_1153095567726851806-A-2025-12-18-17-38-34-214912-c9ilpwzxalt"),
    ("8qni", "af-q13191-f1", "af-q13191-f1_out_-184824251272106742", "af-q13191-f1_out_-184824251272106742-A-2025-12-18-18-20-17-843186-5595101188430149814"),
    ("8v81", "af-p13569-f1", "af-p13569-f1_out_5062893454687777811", "af-p13569-f1_out_5062893454687777811-A-2025-12-18-18-23-47-374313-8974034245740134829"),
    ("8jp0", "af-p32418-f1", "af-p32418-f1_out_-121382155796052104", "af-p32418-f1_out_-121382155796052104-A-2025-12-18-19-05-07-618763-3016327104239585995"),
    ("7yg5", "af-q15878-f1", "af-q15878-f1_out_8248982329038707866", "af-q15878-f1_out_8248982329038707866-A-2025-12-18-19-16-13-776440-3338984357806762671")
    # ("8uk6", "af-q14191-f1", "af-q14191-f1-A-2025-12-18-14-32-07-067312-gpn757ep4vl"),
):
    path = f"../../../AlloPockets/gradio/{jobid}/{jobdir}"
    pockets_out = f"{path}/{jobid}/{jobid}_out" # /{pdb_id}_out.cif
    news[holo] = dict(
        pdb_id = pdb_id,
        # path = path,
        predsf = f"{path}/predictions.csv",
        # pockets_out = pockets_out,
        pocketsf = f"{pockets_out}/{jobid}_out.cif",
        pocketsdir = f"{pockets_out}/pockets"
    )

# news["8uk6"] = dict(
#     pdb_id = "AF-A0A1D8PQM9-F1".lower(),
#     # path = path,
#     # predsf = f"{path}/predictions.csv",
#     # pockets_out = pockets_out,
#     pocketsf = f"../../training_data/8.Apos/Extra_set/pockets/AF-A0A1D8PQM9-F1/AF-A0A1D8PQM9-F1_out/AF-A0A1D8PQM9-F1_out.cif",
#     pocketsdir = f"../../training_data/8.Apos/Extra_set/pockets/AF-A0A1D8PQM9-F1/AF-A0A1D8PQM9-F1_out/pockets"
# )

news

{'7gqu': {'pdb_id': 'af-q14191-f1',
  'predsf': '../../../AlloPockets/gradio/af-q14191-f1_out_1153095567726851806/af-q14191-f1_out_1153095567726851806-A-2025-12-18-17-38-34-214912-c9ilpwzxalt/predictions.csv',
  'pocketsf': '../../../AlloPockets/gradio/af-q14191-f1_out_1153095567726851806/af-q14191-f1_out_1153095567726851806-A-2025-12-18-17-38-34-214912-c9ilpwzxalt/af-q14191-f1_out_1153095567726851806/af-q14191-f1_out_1153095567726851806_out/af-q14191-f1_out_1153095567726851806_out.cif',
  'pocketsdir': '../../../AlloPockets/gradio/af-q14191-f1_out_1153095567726851806/af-q14191-f1_out_1153095567726851806-A-2025-12-18-17-38-34-214912-c9ilpwzxalt/af-q14191-f1_out_1153095567726851806/af-q14191-f1_out_1153095567726851806_out/pockets'},
 '8qni': {'pdb_id': 'af-q13191-f1',
  'predsf': '../../../AlloPockets/gradio/af-q13191-f1_out_-184824251272106742/af-q13191-f1_out_-184824251272106742-A-2025-12-18-18-20-17-843186-5595101188430149814/predictions.csv',
  'pocketsf': '../../../AlloPockets/grad

## Sites

In [8]:
with open("../../training_data/7.Extra_set/news_sites.pkl", "rb") as f:
    news_sites = pickle.load(f) # {k: v for k, v in pickle.load(f).items() if k in ("7gqu", "8qni", "8v81", "8jp0", "7yg5")}

len(news_sites), news_sites["7gqu"]

(25,
 [{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
   3420           X1L             D               4            .   
   
        pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
   3420                 ?        1002          X1L            A   
   
        pdbx_PDB_model_num pdbx_label_index pdbx_sifts_xref_db_name  \
   3420                  1             1002                       ?   
   
        pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res  
   3420                      ?                      ?                      ?  ,
   'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
   0            VAL             A               1           55                 ?   
   1            MET             A               1           56                 ?   
   2            ALA             A               1           57                 ?   
   3            THR             A               1          18

In [9]:
news_sites = {
    f"af-{up_id.lower()}-f1": [news_sites[pdb_id][0]["site"][["pdbx_sifts_xref_db_acc", "pdbx_sifts_xref_db_num", "pdbx_sifts_xref_db_res"]],]
    for pdb_id, up_id in ups
}

for pdb_id, up_id in ups:
    if pdb_id == "8jp0":
        up_id = f"af-{up_id.lower()}-f1"
        news_sites[up_id] = [(
            news_sites[up_id][0]
            .replace({"pdbx_sifts_xref_db_acc": {"P32418-2": "P32418"}})
            .loc[news_sites[up_id][0]["pdbx_sifts_xref_db_num"].astype(int) < 800]
        ),]

news_sites

{'af-q14191-f1': [   pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res
  0                  Q14191                    570                      V
  1                  Q14191                    571                      M
  2                  Q14191                    572                      A
  3                  Q14191                    703                      T
  4                  Q14191                    704                      A
  5                  Q14191                    705                      T
  6                  Q14191                    706                      A
  7                  Q14191                    707                      S
  8                  Q14191                    711                      R
  9                  Q14191                    725                      I
  10                 Q14191                    726                      T
  11                 Q14191                    727                      C
  12                 Q

In [10]:
news_sites = {
    k: [cif.residues.merge(v[0]),]
    for k, v in news_sites.items()
    for cif in (Cif(k, f"{k}_out.cif"),)
}

news_sites

{'af-q14191-f1': [   label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
  0            VAL             A               1          570                 ?   
  1            MET             A               1          571                 ?   
  2            ALA             A               1          572                 ?   
  3            THR             A               1          703                 ?   
  4            ALA             A               1          704                 ?   
  5            THR             A               1          705                 ?   
  6            ALA             A               1          706                 ?   
  7            SER             A               1          707                 ?   
  8            ARG             A               1          711                 ?   
  9            ILE             A               1          725                 ?   
  10           THR             A               1          726          

# Our results

In [11]:
model5_results = {
    holod["pdb_id"]: {
        pocket["Pocket"]: {
            "prob": pocket["Allosteric score"],
            "residues": Pocket(f'{holod["pocketsdir"]}/{pocket["Pocket"]}_atm.cif').residues
        }
        for i, pocket in pd.read_csv(holod["predsf"]).iterrows()
    }
    for holo, holod in news.items()

}

model5_results

{'af-q14191-f1': {'pocket1': {'prob': 0.6842876076698303,
   'residues':     label_comp_id label_asym_id label_seq_id pdbx_PDB_ins_code auth_seq_id  \
   0             ILE             A         1177                 ?        1177   
   2             GLN             A          608                 ?         608   
   4             ALA             A          831                 ?         831   
   6             ASN             A          829                 ?         829   
   7             ALA             A         1176                 ?        1176   
   ..            ...           ...          ...               ...         ...   
   177           TYR             A          839                 ?         839   
   182           GLN             A          850                 ?         850   
   187           ILE             A          883                 ?         883   
   196           ALA             A          841                 ?         841   
   204           ALA             A     

In [12]:
model5_results.keys()

dict_keys(['af-q14191-f1', 'af-q13191-f1', 'af-p13569-f1', 'af-p32418-f1', 'af-q15878-f1'])

# Results - old

# Results - new

## All models

In [13]:
defaults = { "all_pockets_in_output": True, "prob_key": "prob" }

models = {
    "model5": { "results": model5_results, **defaults, "labelling": None },
    
    # "allositepro": { "results": allositepro_results, "all_pockets_in_output": False, "prob_key": "hitScore" },
    # "stingallo": { "results": stingallo_results, "all_pockets_in_output": False, "prob_key": None },
    # "allofusion": { "results": allofusion_results, "all_pockets_in_output": False, "prob_key": None },
    
    # "passer_ensemble": { "results": passer_results["ensemble"], **defaults, "prob_key": "prob/score" },
    # "passer_automl": { "results": passer_results["automl"], **defaults, "prob_key": "prob/score" },
    # "passer_rank": { "results": passer_results["rank"], **defaults, "prob_key": "prob/score" },
    # "deepallo": { "results": deepallo_results, **defaults },
    # "alloses": { "results": alloses_results, **defaults, "prob_key": "pro_ave" },
    # "mefallosite": { "results": mefallosite_results, **defaults },
    # "allo": { "results": allo_results, **defaults },
}
models

{'model5': {'results': {'af-q14191-f1': {'pocket1': {'prob': 0.6842876076698303,
     'residues':     label_comp_id label_asym_id label_seq_id pdbx_PDB_ins_code auth_seq_id  \
     0             ILE             A         1177                 ?        1177   
     2             GLN             A          608                 ?         608   
     4             ALA             A          831                 ?         831   
     6             ASN             A          829                 ?         829   
     7             ALA             A         1176                 ?        1176   
     ..            ...           ...          ...               ...         ...   
     177           TYR             A          839                 ?         839   
     182           GLN             A          850                 ?         850   
     187           ILE             A          883                 ?         883   
     196           ALA             A          841                 ?         8

# Labelling - new

In [14]:
# Percentage of residues of "one" in "other"
get_overlap = lambda one, other: (
    len( one.merge(other) ) / len(one)
)

def get_label(overlaps, site_in_pocket=None, pocket_in_site=None):
    assert not (site_in_pocket==None and pocket_in_site==None)
    
    if site_in_pocket is None:
        return int( overlaps["pocket_in_site"] >= pocket_in_site )
    if pocket_in_site is None:
        return int( overlaps["site_in_pocket"] >= site_in_pocket )
    return int( overlaps["site_in_pocket"] >= site_in_pocket or overlaps["pocket_in_site"] >= pocket_in_site )

In [15]:
# get_overlaps = lambda pdb, pocketd: {
#     name: get_overlap(*one_in_other) 
#         for site in news_sites[pdb]
#             for name, one_in_other in (
#                 ("pocket_in_site", (pocketd["residues"], site["site"])),
#                 ("site_in_pocket", (site["site"], pocketd["residues"])),
#             )
# }

get_overlaps = lambda pdb, pocketd: max(
    (
        {
            name: get_overlap(*one_in_other)
            for name, one_in_other in (
                ("pocket_in_site", (pocketd["residues"], site)),
                ("site_in_pocket", (site, pocketd["residues"])),
            )
        }
        for site in news_sites[pdb]
    ),
    key=lambda x: x["site_in_pocket"]
)



# get_overlaps_special = lambda pdb, pocketd: {
#     name: get_overlap(*one_in_other) 
#         for pocketres in (pocketd["residues"],)
#             for sited in news_sites[pdb]
#                 for site in (sited["site"].query(f"auth_asym_id == '{pocketres.auth_asym_id.unique().item()}'"),)
#                     for name, one_in_other in (
#                         ("pocket_in_site", (pocketres, site)),
#                         ("site_in_pocket", (site, pocketres)),
#                     )
# }

get_overlaps_special = lambda pdb, pocketd: max(
    (
        {
            name: get_overlap(*one_in_other)
            for name, one_in_other in (
                ("pocket_in_site", (pocketres, site)),
                ("site_in_pocket", (site, pocketres)),
            )
        }
        for pocketres in (pocketd["residues"],)
            for sitei in news_sites[pdb]
                for site in (sitei.query(f"auth_asym_id == '{pocketres.auth_asym_id.unique().item()}'"),)
    ),
    key=lambda x: x["site_in_pocket"]
)

In [16]:
# special_7sns_sc will never be used
def label_results(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None, topx=False, special_7sns_sc=False):
    df = pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
            # "pred": pocketd["pred"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in ((
            get_overlaps_special(pdb, pocketd)
            if special_7sns_sc and pdb == "7sns"
            else get_overlaps(pdb, pocketd)
        ),)
    ))
    
    if prob_key is not None and topx:
        df["pred"] = (
            # Start from a Series where each value/row (sample/pocket) is the total number of pos. labels on its PDB
            df.groupby("pdb")["label"].transform("sum")
            # Then subtract this "total num. of pos. in a PDB" by the rank of each pocket in a PDB, sorted by the probability
            .sub(df.groupby("pdb")["prob"].rank(method="first", ascending=False))
            # If the subtraction is positive or 0 it means that the pocket is in the topX and will be assigned 1
            >= 0
        ).astype(int)
        # If a PDB has no + labelled pocket, assign the highest prob. as positive
        for pdb, group in df.groupby("pdb"):
            if group["pred"].sum() == 0:
                df.loc[ group["prob"].idxmax(), "pred" ] = 1
    
    return df.sort_values("max_overlap", ascending=False)

## Model 5.

In [17]:
labelled_results = label_results(model5_results, site_in_pocket=0.35, pocket_in_site=None, topx=True, prob_key="prob")#.iloc[:40]#.sort_values("label", ascending=False).iloc[:40] # , prob_key=models["allo"]["prob_key"]
labelled_results.iloc[:40]

,pdb,pocket,prob,label,max_overlap,pocket_in_site,site_in_pocket,pred
53,af-p13569-f1,pocket3,1.788500e-01,1,0.888889,0.489796,0.888889,0
15,af-q14191-f1,pocket21,3.319631e-04,1,0.846154,0.846154,0.379310,0
42,af-q13191-f1,pocket1,0.000000e+00,1,0.666667,0.392157,0.666667,0
179,af-q15878-f1,pocket21,4.279893e-04,0,0.666667,0.666667,0.206897,0
9,af-q14191-f1,pocket5,9.369624e-04,0,0.600000,0.600000,0.310345,0
162,af-q15878-f1,pocket45,4.124008e-03,1,0.592593,0.592593,0.551724,0
135,af-p32418-f1,pocket4,1.805010e-05,0,0.461538,0.461538,0.300000,0
45,af-q13191-f1,pocket4,0.000000e+00,0,0.363636,0.363636,0.133333,0
113,af-p32418-f1,pocket24,5.762792e-01,1,0.350000,0.145833,0.350000,1
189,af-q15878-f1,pocket8,1.230930e-04,0,0.333333,0.333333,0.241379,0


In [18]:
labelled_results.sort_values("prob", ascending=False).iloc[:40]

,pdb,pocket,prob,label,max_overlap,pocket_in_site,site_in_pocket,pred
52,af-p13569-f1,pocket1,1.000000,0,0.148148,0.011019,0.148148,1
144,af-q15878-f1,pocket2,0.889049,0,0.000000,0.000000,0.000000,1
145,af-q15878-f1,pocket1,0.868856,0,0.206897,0.077922,0.206897,0
0,af-q14191-f1,pocket1,0.684288,0,0.137931,0.065574,0.137931,1
113,af-p32418-f1,pocket24,0.576279,1,0.350000,0.145833,0.350000,1
114,af-p32418-f1,pocket31,0.575299,0,0.000000,0.000000,0.000000,0
146,af-q15878-f1,pocket85,0.547140,0,0.000000,0.000000,0.000000,0
147,af-q15878-f1,pocket29,0.362232,0,0.034483,0.022222,0.034483,0
115,af-p32418-f1,pocket25,0.334627,0,0.000000,0.000000,0.000000,0
1,af-q14191-f1,pocket11,0.230360,0,0.000000,0.000000,0.000000,0


In [19]:
labelled_results.sort_values(["pred", "prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,label,max_overlap,pocket_in_site,site_in_pocket,pred
52,af-p13569-f1,pocket1,1.000000,0,0.148148,0.011019,0.148148,1
144,af-q15878-f1,pocket2,0.889049,0,0.000000,0.000000,0.000000,1
0,af-q14191-f1,pocket1,0.684288,0,0.137931,0.065574,0.137931,1
113,af-p32418-f1,pocket24,0.576279,1,0.350000,0.145833,0.350000,1
40,af-q13191-f1,pocket10,0.000005,0,0.000000,0.000000,0.000000,1
145,af-q15878-f1,pocket1,0.868856,0,0.206897,0.077922,0.206897,0
114,af-p32418-f1,pocket31,0.575299,0,0.000000,0.000000,0.000000,0
146,af-q15878-f1,pocket85,0.547140,0,0.000000,0.000000,0.000000,0
147,af-q15878-f1,pocket29,0.362232,0,0.034483,0.022222,0.034483,0
115,af-p32418-f1,pocket25,0.334627,0,0.000000,0.000000,0.000000,0


In [20]:
models["model5"]["labelling"] = {"site_in_pocket": 0.35, "pocket_in_site": None}

In [36]:
pd.to_pickle(models, "models_lenient_labelling.pkl")

# Labelling - old

# Scoring

## Original

## TopX

In [32]:
models_preds = {}

for model, modeld in models.items():
    results = modeld["results"]
    if modeld["labelling"] is not None:
        labelled_results = label_results(results, **modeld["labelling"], prob_key=modeld["prob_key"], topx=True)
    else:
        labelled_results = pd.DataFrame((
            {
                "pdb": pdb,
                "pocket": pocket,
                **pocketd
            }
            for pdb, pockets in results.items()
            for pocket, pocketd in pockets.items()
        ))

    preds = {}
    for pdb in news_sites:
        # if pdb == "4jqi": continue #######################################################################################
        pdbpreds = labelled_results.query(f"pdb == '{pdb}'")
        total = len(pdbpreds)
        if total > 0:
            # # Top1 pred for our model
            # if model == "model5":
            #     pdbpreds.loc[:, "pred"] = pdbpreds[["prob"]].apply(lambda x: (x == x.max()).astype(int)).values
            # elif model == "allofusion":
            #     pdbpreds = pdbpreds.drop(
            #         pdbpreds.loc[lambda x: (x.pred == 1) & (x.label == 1)]
            #         .index[1:]
            #     )

            # # if pdb is 7sns, if there's more than 1 label 1, keep only the pred one if exists, or just 1 in general
            # if pdb == "7sns":
            #     pdbpreds = pdbpreds.drop(
            #         pdbpreds.loc[pdbpreds.label == 1]
            #         .sort_values("pred", ascending=False)
            #         .index[1:]
            #     )
                
            tp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 1)] )
            fp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 0)] )
            fn = len( pdbpreds.loc[lambda x: (x["pred"] == 0) & (x["label"] == 1)] )
            if tp + fn == 0:
                fn = 1 # There's at least 1 allo. pocket per PDB
        else:
            tp, fp = 0, 0
            fn = 1 # There's at least 1 allo. pocket per PDB

        if total == 0 or not modeld["all_pockets_in_output"]:
            preds[pdb] = {
                "tp": tp,
                "fn": fn,
                "fp": fp
            }
        else:
            preds[pdb] = {
                "total": total,
                "tp": tp,
                "fn": fn,
                "fp": fp
            }

    models_preds[model] = preds

models_preds

{'model5': {'af-q14191-f1': {'total': 40, 'tp': 0, 'fn': 1, 'fp': 1},
  'af-q13191-f1': {'total': 12, 'tp': 0, 'fn': 1, 'fp': 1},
  'af-p13569-f1': {'total': 61, 'tp': 0, 'fn': 1, 'fp': 1},
  'af-p32418-f1': {'total': 31, 'tp': 1, 'fn': 0, 'fp': 0},
  'af-q15878-f1': {'total': 85, 'tp': 0, 'fn': 1, 'fp': 1}}}

In [33]:
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix

In [34]:
models_metrics = {}

for model, preds in models_preds.items():
    df = pd.DataFrame(preds).T
    
    if models[model]["all_pockets_in_output"]:
        y_true = [1] * df["tp"].sum() + [1] * df["fn"].sum() + [0] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())
        y_pred = [1] * df["tp"].sum() + [0] * df["fn"].sum() + [1] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())

        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "macro-f1": f1_score(y_true, y_pred, average="macro"),
            "confmat": pd.DataFrame(confusion_matrix(y_true, y_pred))
        }
        
    else:
        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
        }

models_metrics = pd.DataFrame(models_metrics).T.sort_values(["tp", "mcc"], ascending=False)
models_metrics

,model,tp,fn,fp,mcc,macro-f1,confmat
model5,model5,1,4,4,0.182143,0.591071,0 1 0 220 4 1 4 1


In [35]:
models_metrics.infer_objects().to_csv("models_metrics_topx.csv", index=False, sep="\t", decimal=",")

# Old

In [63]:
models_preds = {}

for model, modeld in models.items():
    results = modeld["results"]
    if "model5" not in model:
        labelled_results = label_results_topx(results, **modeld["labelling"], prob_key=modeld["prob_key"])
    else:
        labelled_results = label_our_results_topx(results, **modeld["labelling"])

    preds = {}
    for pdb in news:
        pdbpreds = labelled_results.query(f"pdb == '{pdb}'")
        total = len(pdbpreds)
        if total > 0:
            if model == "allofusion":
                pdbpreds = pdbpreds.drop(
                    pdbpreds.loc[lambda x: (x.pred == 1) & (x.label == 1)]
                    .index[1:]
                )
            tp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 1)] )
            fp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 0)] )
            fn = len( pdbpreds.loc[lambda x: (x["pred"] == 0) & (x["label"] == 1)] )
            if tp + fn == 0:
                fn = 1 # There's at least 1 allo. pocket per PDB
        else:
            tp, fp = 0, 0
            fn = 1 # There's at least 1 allo. pocket per PDB

        if total == 0 or not modeld["all_pockets_in_output"]:
            preds[pdb] = {
                "tp": tp,
                "fn": fn,
                "fp": fp
            }
        else:
            preds[pdb] = {
                "total": total,
                "tp": tp,
                "fn": fn,
                "fp": fp
            }

    models_preds[model] = preds

models_preds

{'model5': {'8sgj': {'total': 28, 'tp': 1, 'fn': 0, 'fp': 0},
  'AF-A0A1D8PQM9-F1': {'total': 30, 'tp': 1, 'fn': 0, 'fp': 0},
  '7l6r': {'total': 14, 'tp': 1, 'fn': 0, 'fp': 0},
  '8vw5': {'total': 13, 'tp': 1, 'fn': 0, 'fp': 0},
  '6yhr': {'total': 20, 'tp': 1, 'fn': 0, 'fp': 0},
  '7xlq': {'total': 69, 'tp': 0, 'fn': 1, 'fp': 1},
  '5b0u': {'total': 2, 'tp': 0, 'fn': 1, 'fp': 1},
  '5uak': {'total': 75, 'tp': 1, 'fn': 0, 'fp': 0},
  '4jqi': {'total': 19, 'tp': 0, 'fn': 1, 'fp': 1}},
 'allositepro': {'8sgj': {'tp': 0, 'fn': 1, 'fp': 0},
  'AF-A0A1D8PQM9-F1': {'tp': 0, 'fn': 1, 'fp': 0},
  '7l6r': {'tp': 1, 'fn': 0, 'fp': 0},
  '8vw5': {'tp': 0, 'fn': 1, 'fp': 0},
  '6yhr': {'tp': 0, 'fn': 1, 'fp': 0},
  '7xlq': {'tp': 0, 'fn': 1, 'fp': 0},
  '5b0u': {'tp': 0, 'fn': 1, 'fp': 1},
  '5uak': {'tp': 0, 'fn': 1, 'fp': 0},
  '4jqi': {'tp': 0, 'fn': 1, 'fp': 0}},
 'stingallo': {'8sgj': {'tp': 0, 'fn': 1, 'fp': 0},
  'AF-A0A1D8PQM9-F1': {'tp': 0, 'fn': 1, 'fp': 0},
  '7l6r': {'tp': 0, 'fn': 1,

In [64]:
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix

In [65]:
models_metrics = {}

for model, preds in models_preds.items():
    df = pd.DataFrame(preds).T
    
    if models[model]["all_pockets_in_output"]:
        y_true = [1] * df["tp"].sum() + [1] * df["fn"].sum() + [0] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())
        y_pred = [1] * df["tp"].sum() + [0] * df["fn"].sum() + [1] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())

        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "macro-f1": f1_score(y_true, y_pred, average="macro"),
            "confmat": pd.DataFrame(confusion_matrix(y_true, y_pred))
        }
        
    else:
        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
        }

models_metrics = pd.DataFrame(models_metrics).T.sort_values(["tp", "mcc"], ascending=False)
models_metrics

,model,tp,fn,fp,mcc,macro-f1,confmat
model5,model5,6,3,3,0.655172,0.827586,0 1 0 258 3 1 3 6
mefallosite,mefallosite,4,5,5,0.432396,0.716198,0 1 0 410 5 1 5 4
allo,allo,4,5,5,0.42108,0.71054,0 1 0 209 5 1 5 4
passer_automl,passer_automl,3,6,6,0.320885,0.660443,0 1 0 476 6 1 6 3
alloses,alloses,3,6,6,0.320885,0.660443,0 1 0 476 6 1 6 3
allofusion,allofusion,3,6,55,NaN,NaN,NaN
passer_ensemble,passer_ensemble,2,7,7,0.207699,0.60385,0 1 0 475 7 1 7 2
passer_rank,passer_rank,2,7,7,0.207699,0.60385,0 1 0 475 7 1 7 2
deepallo,deepallo,2,7,7,0.207423,0.603712,0 1 0 466 7 1 7 2
allositepro,allositepro,1,8,1,NaN,NaN,NaN


In [66]:
models_metrics.infer_objects().to_csv("models_metrics_top1_apos_lenient_labelling.csv", index=False, sep=";", decimal=",")